# 🔄 Transfer Learning - Theory & Strategies

Welcome to the **Advanced Techniques** module! We start with one of the most practically useful techniques in modern Deep Learning: **Transfer Learning**.

In the early days of neural networks, every model was trained from scratch. Weights were initialized randomly, and the model had to learn how to detect edges, textures, and shapes from zero. This required massive datasets and weeks of training.

**Transfer Learning** changes the game. Ideally, a model trained on a massive dataset (like ImageNet, with 1.2M images) has already learned a rich representation of the visual world. Instead of discarding this knowledge, we **transfer** it to a new, specific task (like classifying types of flowers or detecting defects in metal).

### In this notebook, we will cover:
1.  **The Intuition:** Why transfer learning works.
2.  **The Two Main Strategies:** Feature Extraction vs. Fine-Tuning.
3.  **The Decision Matrix:** How to choose the right strategy based on your data.
4.  **Implementation Mechanics:** How to freeze layers, replace heads, and manage learning rates in PyTorch.

## 1. The Intuition

Imagine you want to learn how to ride a motorcycle.
* **Approach A (From Scratch):** You have never seen a wheel, a road, or a vehicle before. You have to learn physics, balance, and traffic rules from zero.
* **Approach B (Transfer Learning):** You already know how to ride a bicycle. You transfer your knowledge of balance and steering, and only need to learn the specific "delta" (how to use the engine/clutch).

In Deep Learning:
* **Source Task:** Usually a large-scale dataset (e.g., ImageNet for vision, Wikipedia for NLP).
* **Target Task:** Your specific problem (e.g., Medical X-Ray classification).

**Why does it work?**
Deep Neural Networks learn hierarchical features:
* **Early Layers:** Detect generic features (edges, corners, color blobs). These are useful for almost *any* visual task.
* **Middle Layers:** Detect combinations of features (textures, simple shapes).
* **Later Layers:** Detect task-specific semantic features (dog ears, car wheels).

When we use Transfer Learning, we reuse the generic early layers and simply retrain or replace the later layers.

## 2. Setup

We will use `torchvision` to inspect standard models and understand how to manipulate them.

In [1]:
import torch
import torch.nn as nn
from torchvision import models

# Check for device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

Using device: cuda


## 3. Strategy 1: Fixed Feature Extractor

In this strategy, we treat the pre-trained network as an arbitrary feature extractor.

1.  **Load** a pre-trained model (e.g., ResNet18).
2.  **Freeze** all the weights (stop gradient calculation).
3.  **Remove** the final fully connected layer (the "head").
4.  **Replace** it with a new, untrained head that matches our number of classes.
5.  **Train** *only* the new head.

**When to use:** When your dataset is small and very similar to the pre-training dataset.

In [4]:
def setup_feature_extraction(num_classes=10):
    # 1. Load Pre-trained Model
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    # 2. Freeze all parameters
    # We loop through parameters and set requires_grad to False.
    # This prevents PyTorch from calculating gradients for these layers during backprop.
    for param in model.parameters():
        param.requires_grad = False

    # 3. Replace the Head
    # In ResNet, the final layer is called 'fc' (Fully Connected).
    # We need to know the size of the features coming INTO this layer.
    num_ftrs = model.fc.in_features

    print(f"Original Head: {model.fc}")
    print(f"Input features to head: {num_ftrs}")

    # We replace it with a new Linear layer.
    # By default, new layers have requires_grad=True.
    model.fc = nn.Linear(num_ftrs, num_classes)

    return model

# Initialize
model_extractor = setup_feature_extraction(num_classes=5)

# Verify Freezing
print("\n--- Verifying Frozen Status ---")
for name, param in list(model_extractor.named_parameters())[0:3]: # Check first few layers
    print(f"{name}: requires_grad={param.requires_grad}")

print(f"fc.weight: requires_grad={model_extractor.fc.weight.requires_grad}") # Check new head

Original Head: Linear(in_features=512, out_features=1000, bias=True)
Input features to head: 512

--- Verifying Frozen Status ---
conv1.weight: requires_grad=False
bn1.weight: requires_grad=False
bn1.bias: requires_grad=False
fc.weight: requires_grad=True


## 4. Strategy 2: Fine-Tuning

In this strategy, we optimize the weights of the pre-trained model along with the new head. We are "tweaking" the pre-learned features to adapt to the new task.

1.  **Load** pre-trained model.
2.  **Replace** the head.
3.  **Unfreeze** (or keep unfrozen) the rest of the model.
4.  **Train** the whole network.

**Crucial Detail:** Because the pre-trained weights are already very good, we don't want to change them drastically. We usually use a **very low learning rate** for the backbone (e.g., 1e-4 or 1e-5) and a higher learning rate for the new head.

**When to use:** When your dataset is large, or when the domain differs significantly from the source dataset.

In [5]:
def setup_fine_tuning(num_classes=10):
    # 1. Load Model
    model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

    # 2. Do NOT freeze weights (default is requires_grad=True)

    # 3. Replace Head
    num_ftrs = model.fc.in_features
    model.fc = nn.Linear(num_ftrs, num_classes)

    return model

model_finetune = setup_fine_tuning()

# --- Differential Learning Rates ---
# In fine-tuning, we often want the backbone to learn SLOWLY and the head to learn FAST.
# We can do this by passing parameter groups to the optimizer.

optimizer = torch.optim.Adam([
    # Group 1: The Backbone (Pre-trained layers) -> Low LR
    {'params': [p for n, p in model_finetune.named_parameters() if "fc" not in n], 'lr': 1e-5},

    # Group 2: The Head (New layer) -> High LR
    {'params': model_finetune.fc.parameters(), 'lr': 1e-3}
])

print("Optimizer configured with differential learning rates.")

Optimizer configured with differential learning rates.


## 5. The Decision Matrix

How do you choose between Feature Extraction and Fine-Tuning? It depends on two factors:
1.  **Size of your Target Dataset** (Small vs. Large).
2.  **Similarity to Source Dataset** (Similar vs. Different).

| Scenario | Data Size | Data Similarity | Strategy | Explanation |
| :--- | :--- | :--- | :--- | :--- |
| **1** | Small | Similar | **Feature Extraction** | Your data is small (risk of overfitting) but similar to ImageNet. The CNN already knows how to recognize these features. Just train the classifier. |
| **2** | Large | Similar | **Fine-Tuning** | You have enough data to tune the weights without overfitting. You can fine-tune to squeeze out extra accuracy. |
| **3** | Small | Different | **Feature Extraction (Partial)** | Tricky. The high-level features (top of network) might not be relevant, but low-level (edges) are. Freeze the bottom layers, train the top layers + classifier. |
| **4** | Large | Different | **Fine-Tuning / From Scratch** | Since the domain is different (e.g., medical scans vs. dogs/cats), the ImageNet weights aren't perfect, but you have enough data to retrain the whole network. Starting with weights is still better than random. |

## 6. Input Standardization: A Common Pitfall

When using pre-trained models (especially from `torchvision`), you **must** use the same normalization statistics that the model was trained with.

For ImageNet models, these values are standard:
* **Mean:** `[0.485, 0.456, 0.406]`
* **Std:** `[0.229, 0.224, 0.225]`

If you feed raw images (0-1) or normalize incorrectly, the model's activations will be skewed, and performance will suffer.

In [6]:
from torchvision import transforms

# The standard transform for Transfer Learning inference
preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224), # Most CNNs expect 224x224
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

print("Preprocessing pipeline created.")

Preprocessing pipeline created.


## 7. Advanced: Freezing Specific Layers

Sometimes (Scenario 3 in our matrix), you want to keep the generic low-level features (edges) but retrain the high-level features (shapes).

In ResNet, layers are grouped into blocks: `layer1`, `layer2`, `layer3`, `layer4`.

In [7]:
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

# Freeze everything first
for param in model.parameters():
    param.requires_grad = False

# Unfreeze just the last block (layer4) and the fc layer
for name, param in model.named_parameters():
    if "layer4" in name or "fc" in name:
        param.requires_grad = True

print("--- Partial Unfreezing Check ---")
print(f"Layer 1 (conv1): {model.layer1[0].conv1.weight.requires_grad}") # Should be False
print(f"Layer 4 (conv1): {model.layer4[0].conv1.weight.requires_grad}") # Should be True
print(f"FC Layer: {model.fc.weight.requires_grad}")       # Should be True

--- Partial Unfreezing Check ---
Layer 1 (conv1): False
Layer 4 (conv1): True
FC Layer: True


## 8. Conclusion

Transfer learning allows us to stand on the shoulders of giants. By reusing weights trained on millions of images, we can:
1.  Train models on very small datasets (e.g., 50 images per class).
2.  Converge much faster (fewer epochs).
3.  Achieve higher accuracy than training from scratch.

In the next notebook, we will apply "Strategy 1" to build a practical image classifier on a custom dataset.